# 说明：

1. 按照跟踪列表
2. 确定投资时间段的股票收益率
3. 跟踪这个时间段关键基本面指标的变化
4. 输出到excel文件

# 1、确定跟踪的列表

In [167]:
import numpy as np
import pandas as pd

import warnings
# 忽略所有警告
warnings.filterwarnings('ignore')

In [168]:
sec_list = [
    "000002.XSHE",	#万科A
    "000333.XSHE",	#美的集团
    "000672.XSHE",	#上峰水泥
    "000830.XSHE",	#鲁西化工
    "002001.XSHE",	#新和成
    "002508.XSHE",	#老板电器
    "002597.XSHE",	#金禾实业
    "002677.XSHE",	#浙江美大
    "002884.XSHE",	#凌霄泵业
    "300003.XSHE",	#乐普医疗
    "300607.XSHE",	#拓斯达
    "300676.XSHE",	#华大基因
    "600031.XSHG",	#三一重工
    "600352.XSHG",	#浙江龙盛
    "600383.XSHG",	#金地集团
    "600585.XSHG",	#海螺水泥
    "603699.XSHG",	#纽威股份
    "002142.XSHE",	#宁波银行
    "600036.XSHG",	#招商银行
    "600030.XSHG",	#中信证券
    "000776.XSHE",	#广发证券
    "601318.XSHG",	#中国平安
    "601601.XSHG", 	#中国太保
    "159919.XSHE"
]

In [169]:
sec_list

['000002.XSHE',
 '000333.XSHE',
 '000672.XSHE',
 '000830.XSHE',
 '002001.XSHE',
 '002508.XSHE',
 '002597.XSHE',
 '002677.XSHE',
 '002884.XSHE',
 '300003.XSHE',
 '300607.XSHE',
 '300676.XSHE',
 '600031.XSHG',
 '600352.XSHG',
 '600383.XSHG',
 '600585.XSHG',
 '603699.XSHG',
 '002142.XSHE',
 '600036.XSHG',
 '600030.XSHG',
 '000776.XSHE',
 '601318.XSHG',
 '601601.XSHG',
 '159919.XSHE']

In [170]:
# 所有价格的序列

start_date = "2021-1-1"
end_date = "2025-8-2"


# 2、确定收益率差异

In [171]:
price_sec = get_price(sec_list, start_date=start_date, end_date=end_date, frequency='daily',fq='pre', fields=['close'])['close']


In [172]:
yield_dict = {}

for sec in price_sec.columns:
    yield_dict[sec] = price_sec[sec][-1]/price_sec[sec][0]
    
    
yield_pd = pd.DataFrame(yield_dict,index=[0]).T
yield_pd.columns = ['sec_yield']

In [173]:
yield_pd['bench_compare'] = yield_pd['sec_yield'] - yield_pd['sec_yield']['159919.XSHE']

In [174]:
# 名字信息
name_pd =pd.DataFrame(get_all_securities()['display_name'])


# 提取行业的编码信息
Sec_list = [[d[0],d[1]['industry_code'],d[1]['industry_name']] for d in pd.DataFrame.from_dict(get_industry(sec_list, date=end_date), orient='index')['sw_l1'].items()]

# 重新生成新的pandas数据，准备进行合并
industry_pd = pd.DataFrame(data = Sec_list,columns = ['code','industry','industry_name'])
industry_pd = industry_pd.set_index(['code'])

In [175]:
# 增加名字
yield_pd = pd.concat([yield_pd,name_pd], axis =1,sort=False,join = 'inner')

# 增加行业
yield_pd = pd.concat([yield_pd,industry_pd], axis =1,sort=False,join = 'inner')

In [176]:
yield_pd

,sec_yield,bench_compare,display_name,industry,industry_name
000002.XSHE,0.271568,-0.576284,万科A,801180,房地产I
000333.XSHE,0.864663,0.016811,美的集团,801110,家用电器I
000672.XSHE,0.688281,-0.159571,上峰水泥,801710,建筑材料I
000830.XSHE,1.165323,0.317471,鲁西化工,801030,化工I
002001.XSHE,1.063576,0.215723,新和成,801030,化工I
002508.XSHE,0.538031,-0.309821,老板电器,801110,家用电器I
002597.XSHE,0.701743,-0.146109,金禾实业,801030,化工I
002677.XSHE,0.533742,-0.314110,浙江美大,801110,家用电器I
002884.XSHE,1.264019,0.416167,凌霄泵业,801890,机械设备I
300003.XSHE,0.637139,-0.210713,乐普医疗,801150,医药生物I


In [177]:
## 输出到文件
# yield_pd.to_csv('yield.csv')

# 3、跟踪基本面指标变化


In [178]:
# 因子列表
factor_pool = [
        'book_to_price_ratio',    # PB值的倒数
        "earnings_to_price_ratio", # PE值的倒数
        "predicted_earnings_to_price_ratio",  #预测PE值的倒数
        "net_profit_growth_rate",  # 净利润增长率
        "PEG",                     # PE/G

        "roe_ttm",                 # 过去12个月（TTM）的股东回报率
        "net_profit_ratio",        # 净利率
        "total_asset_turnover_rate", # 资产周转率
        "debt_to_asset_ratio",     # 杠杆比率
        "asset_turnover_ttm",      # 经营资产周转率

        "cash_rate_of_sales",       # 经营活动产生的现金流量净额（TTM） / 营业收入（TTM）
        "net_non_operating_income_to_total_profit",   # 非经营性利润占比
        "operating_tax_to_operating_revenue_ratio_ttm", # 销售税金比例，营业税金及附加（TTM）/营业收入（TTM）
        "invest_income_associates_to_total_profit",    # 对联营和营公司投资收益/利润总额
        "OperatingCycle",          # 应收账款周转天数+存货周转天数
        "DSRI",                     # 本期(年报)应收账款占营业收入比例/上期(年报)应收账款占营业收入比例
        "accounts_payable_turnover_days", # 应付账款周转天数
    
        "net_operating_cash_flow_coverage", # 经营活动现金流占利润的比例
        "cfo_to_ev",                        # 经营活动产生的现金流量净额与企业价值之比TTM
        "quick_ratio",                      # 速动比率=(流动资产合计-存货)/ 流动负债合计
        "fixed_asset_ratio",                # 固定资产比率=(固定资产+工程物资+在建工程)/总资产
        "intangible_asset_ratio",           # 无形资产比率=(无形资产+研发支出+商誉)/总资产
        "SGAI",                             # 本期(年报)销售管理费用占营业收入的比例/上期(年报)销售管理费用占营业收入的比例
    
        "operating_revenue_growth_rate",    # 营业收入增长率=（今年营业收入（TTM）/去年营业收入（TTM））-1
        "financing_cash_growth_rate",       # 过去12个月的筹资现金流量净额 / 4季度前的12个月的筹资现金流量净额 - 1
        "net_operate_cashflow_growth_rate"  # 经营活动产生的现金流量净额增长率	
]


In [179]:
# 时间列表
# 所有价格的序列

# 从2020年起始点的前三年开始计算，方便核实3年前、5年后的情况
start_date = "2017-1-1"
end_date = "2025-8-2"


date_pd = pd.DataFrame(get_price(sec_list, start_date=start_date, end_date=end_date, frequency='daily',fq='pre', fields=['close'])['close'])

date_pd['year'] = date_pd.index.year

july_first_list = date_pd[date_pd.index.month == 7].groupby('year').head(1).index

In [180]:
july_first_list

DatetimeIndex(['2017-07-03', '2018-07-02', '2019-07-01', '2020-07-01',
               '2021-07-01', '2022-07-01', '2023-07-03', '2024-07-01',
               '2025-07-01'],
              dtype='datetime64[ns]', freq=None)

In [181]:
# 导入函数库
from jqfactor import get_factor_values

# 确定要分析的股票
sec_total_analysis_dict = {}

for sec_i in sec_list:
    # 股票数组
    sec_i_analysis_pd = pd.DataFrame(index = factor_pool, columns = july_first_list)

    for date_i in july_first_list:
        factor_data = get_factor_values(securities = sec_i, 
                                    factors=factor_pool, 
                                    start_date=date_i, end_date=date_i)

        for factor_i in factor_pool:
            sec_i_analysis_pd.loc[ factor_i,date_i] =  factor_data[factor_i][sec_i].values[0]
    
    # 汇总股票数据
    sec_total_analysis_dict[sec_i] = sec_i_analysis_pd

In [182]:
sec_total_analysis_dict["000002.XSHE"]

,2017-07-03 00:00:00,2018-07-02 00:00:00,2019-07-01 00:00:00,2020-07-01 00:00:00,2021-07-01 00:00:00,2022-07-01 00:00:00,2023-07-03 00:00:00,2024-07-01 00:00:00,2025-07-01 00:00:00
book_to_price_ratio,-0.186944,-0.180097,-0.291447,0.094253,1.01749,1.69912,2.89095,3.3049,2.56815
earnings_to_price_ratio,0.077,0.112245,0.105106,0.119509,0.143201,0.095551,0.134076,0.119056,-0.730764
predicted_earnings_to_price_ratio,0.097873,0.160031,0.144978,0.159617,0.167499,0.130627,0.141956,0.141632,-0.104614
net_profit_growth_rate,0.072966,0.342759,0.33845,0.072136,0.092335,-0.354113,-0.019605,-0.522493,-4.08862
PEG,0.920451,0.252582,0.467692,0.568695,1.06485,NaN,NaN,NaN,NaN
roe_ttm,0.183246,0.207448,0.215792,0.206721,0.184125,0.095667,0.092006,0.041315,-0.281416
net_profit_ratio,0.115398,0.148466,0.160832,0.14799,0.136934,0.084619,0.073772,0.039126,-0.173512
total_asset_turnover_rate,0.275674,0.208397,0.20322,0.211113,0.227271,0.232725,0.292382,0.313456,0.256887
debt_to_asset_ratio,0.817327,0.840445,0.847734,0.842446,0.813701,0.79803,0.771952,0.72707,0.734756
asset_turnover_ttm,2.07047,0.865265,0.327428,0.440878,0.477082,0.477288,0.617148,0.669064,0.555096


# 4、输出到文件

In [183]:
# 创建 ExcelWriter
with pd.ExcelWriter('fundamental.xlsx') as writer:
    
    # 输出汇总收益率统计数据
    yield_pd.to_excel(writer, sheet_name='yield_total', index=True)
    
    
    for key,value in sec_total_analysis_dict.items():
        df = value
        sheet_name = key
        df.to_excel(writer, sheet_name=sheet_name, index=True)